# Práctica 6 - Preprocesamiento de datos para Machine Learning
**Sistemas de Aprendizaje Automático · Sesión 6**   
**Dataset:** `clientes_ecommerce.csv` · **Modalidad:** individual

## Contexto
Trabajas en una empresa de comercio electrónico que quiere predecir si un cliente comprará el próximo mes. El dataset ya se exploró en las sesiones 4 y 5. Ahora debes transformarlo para que esté listo para entrenar un modelo de clasificación binaria.

**No entrenes modelos en esta práctica.** Completa los bloques `TODO` y responde las preguntas en las celdas Markdown de respuesta.

### Objetivos
- Detectar ausentes y anomalías
- Elegir estrategias de imputación 
- Codificar variables nominales y ordinales
- Comparar escalados
- Separar entrenamiento/prueba
- Construir un `Pipeline` con `ColumnTransformer` 
- Evitar data leakage
- Verificar las matrices resultantes

> Coloca `clientes_ecommerce.csv` en la misma carpeta que este notebook. Usa scikit-learn 1.2 o posterior para `sparse_output=False`.

## Parte 1 - Diagnóstico del dataset
### Ejercicio 1. Carga y estructura
Carga el CSV, muestra 10 filas, dimensiones, tipos de datos, información general y número de registros duplicados.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# TODO: cargar clientes_ecommerce.csv en df
df = pd.read_csv("clientes_ecommerce.csv")

# TODO: mostrar 10 primeras filas
display(df.head(10))

# TODO: consultar shape, dtypes, info y número de duplicados
print("Shape:", df.shape)
print(df.dtypes)
df.info()
print("Duplicados:", df.duplicated().sum())

,cliente_id,edad,antiguedad_meses,ingresos_anuales,gasto_mensual,num_compras,devoluciones,dispositivo,zona,nivel_fidelidad,satisfaccion,compra_proximo_mes
0,20001,67,102.0,9000.00,60.73,7,3,Móvil,Oeste,Oro,5,No
1,20002,28,93.0,35683.86,131.74,6,1,Móvil,Este,Bronce,2,No
2,20003,19,87.0,37152.19,55.33,8,1,Ordenador,Este,Plata,5,Sí
3,20004,55,79.0,40158.29,24.33,7,0,Ordenador,Norte,Bronce,4,Sí
4,20005,39,3.0,27317.06,88.78,5,0,Móvil,Sur,Bronce,4,No
5,20006,45,25.0,32627.01,59.77,9,2,Móvil,Este,Bronce,5,No
6,20007,22,105.0,44101.68,109.67,8,0,Móvil,Norte,Oro,5,Sí
7,20008,39,90.0,44882.10,57.56,7,2,Ordenador,Norte,Plata,2,Sí
8,20009,55,56.0,55299.19,49.94,6,2,Móvil,Sur,Bronce,5,No
9,20010,38,93.0,9000.00,110.69,10,2,Ordenador,Centro,Oro,4,Sí


Shape: (400, 12)
cliente_id              int64
edad                    int64
antiguedad_meses      float64
ingresos_anuales      float64
gasto_mensual         float64
num_compras             int64
devoluciones            int64
dispositivo               str
zona                      str
nivel_fidelidad           str
satisfaccion            int64
compra_proximo_mes        str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cliente_id          400 non-null    int64  
 1   edad                400 non-null    int64  
 2   antiguedad_meses    395 non-null    float64
 3   ingresos_anuales    396 non-null    float64
 4   gasto_mensual       397 non-null    float64
 5   num_compras         400 non-null    int64  
 6   devoluciones        400 non-null    int64  
 7   dispositivo         400 non-null    str    
 8   zona                

**Responde:** ¿Cuántas muestras y columnas hay? ¿Cuál es el target? ¿Qué columna identifica a cada cliente? ¿Hay filas duplicadas?

**Respuesta:**

- 400 muestras y 12 columnas.
- El target es `compra_proximo_mes`.
- `cliente_id` identifica a cada cliente.
- No, hay 0 duplicados.

### Ejercicio 2. Valores ausentes
Calcula el número de ausentes por columna, el porcentaje y el número de filas con al menos un ausente. Completa la tabla.

In [2]:
# TODO: ausentes por columna
ausentes = df.isna().sum()
print(ausentes)

# TODO: porcentaje de ausentes por columna
print((ausentes / len(df) * 100).round(2))

# TODO: filas con algún valor ausente
print("Filas con algún ausente:", df.isna().any(axis=1).sum())

cliente_id            0
edad                  0
antiguedad_meses      5
ingresos_anuales      4
gasto_mensual         3
num_compras           0
devoluciones          0
dispositivo           0
zona                  0
nivel_fidelidad       0
satisfaccion          0
compra_proximo_mes    0
dtype: int64
cliente_id            0.00
edad                  0.00
antiguedad_meses      1.25
ingresos_anuales      1.00
gasto_mensual         0.75
num_compras           0.00
devoluciones          0.00
dispositivo           0.00
zona                  0.00
nivel_fidelidad       0.00
satisfaccion          0.00
compra_proximo_mes    0.00
dtype: float64
Filas con algún ausente: 12


| Variable | Nº ausentes | Porcentaje |
|---|---:|---:|
| `antiguedad_meses` | 5 | 1.25 % |
| `ingresos_anuales` | 4 | 1.00 % |
| `gasto_mensual` | 3 | 0.75 % |

**¿Eliminarías todas las filas incompletas? ¿Por qué?**

Respuesta: No. Son 12 filas (un 3 %) y en cada una solo falta un dato, así que perderíamos el resto de info del cliente. Mejor imputarlas.

### Ejercicio 3. Edad sospechosa
Localiza edades superiores a 120 años, muestra `cliente_id` y `edad` y consulta mínimo y máximo.

In [3]:
# TODO: mínimo y máximo de edad
print("Edad mínima:", df['edad'].min())
print("Edad máxima:", df['edad'].max())

# TODO: localizar registros con edad > 120
# Pista: df.loc[df["edad"] > 120, ["cliente_id", "edad"]]
df.loc[df['edad'] > 120, ['cliente_id', 'edad']]

Edad mínima: 18
Edad máxima: 195


,cliente_id,edad
137,20138,195


**Responde:** ¿Qué harías con esa edad? ¿Por qué no deberías cambiar 195 por 95 sin evidencia?

Respuesta: La marcaría como valor ausente (NaN) y dejaría que se impute luego. Cambiarla a 95 sería inventarse el dato, igual podía ser 19, 59 o cualquier otra cosa.

## Parte 2 - Variables y división
### Ejercicio 4. Clasificación de columnas
Clasifica las columnas según su naturaleza.

| Tipo | Columnas |
|---|---|
| Numéricas | `edad`, `antiguedad_meses`, `ingresos_anuales`, `gasto_mensual`, `num_compras`, `devoluciones`, `satisfaccion` |
| Categóricas nominales | `dispositivo`, `zona` |
| Categóricas ordinales | `nivel_fidelidad` |
| Identificador | `cliente_id` |
| Target | `compra_proximo_mes` |

**Responde:** ¿Por qué `zona` es nominal y `nivel_fidelidad` ordinal? ¿Por qué no incluirías `cliente_id` en el modelo?

Respuesta: Las zonas no tienen orden (Norte no es "más" que Sur), en cambio los niveles sí van de Bronce a Platino. `cliente_id` solo identifica, no dice nada del cliente y el modelo podría aprender patrones que no existen.

### Ejercicio 5. Separar X e y
Excluye de X el identificador y el target. Codifica `No` como 0 y `Sí` como 1. Comprueba dimensiones y distribución de y.

In [4]:
# TODO: X = ... (sin cliente_id ni compra_proximo_mes)
X = df.drop(columns=['cliente_id', 'compra_proximo_mes'])

# TODO: y = ... (mapear No:0, Sí:1)
y = df['compra_proximo_mes'].map({'No': 0, 'Sí': 1})

# TODO: X.shape, y.shape, y.value_counts()
print("X:", X.shape)
print("y:", y.shape)
print(y.value_counts())

X: (400, 10)
y: (400,)
compra_proximo_mes
0    246
1    154
Name: count, dtype: int64


**¿Cuántas características predictoras iniciales hay?**

Respuesta: 10.

### Ejercicio 6. División entrenamiento/prueba
Separa 80 % para entrenamiento y 20 % para prueba. Utiliza `random_state=42` y `stratify=y`. Comprueba tamaños.

In [5]:
# TODO: completar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TODO: mostrar las dimensiones de los cuatro conjuntos
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (320, 10)
X_test: (80, 10)
y_train: (320,)
y_test: (80,)


**Responde:** ¿Cuántas muestras hay en cada conjunto? ¿Para qué sirve `stratify=y`? ¿Por qué dividimos antes de ajustar transformaciones?

Respuesta: 320 en entrenamiento y 80 en prueba. `stratify=y` hace que los dos conjuntos tengan el mismo porcentaje de 0 y 1 que el original (≈ 38.5 % de `Sí`). Dividimos antes porque si no las medianas, medias, etc. se calcularían también con la prueba, y eso es data leakage.

## Parte 3 - Imputación y encoding
### Ejercicio 7. Validación de edad
Sobre **copias** de `X_train` y `X_test`, sustituye edades superiores a 120 por `np.nan`. No imputes todavía: eso se hará en el pipeline. El umbral es una regla didáctica de validación del dominio.

In [6]:
X_train = X_train.copy()
X_test = X_test.copy()

# TODO: marcar edades > 120 como NaN en ambos conjuntos
X_train['edad'] = X_train['edad'].mask(X_train['edad'] > 120)
X_test['edad'] = X_test['edad'].mask(X_test['edad'] > 120)

# TODO: contar ausentes en edad en entrenamiento y prueba
print("Ausentes en edad (train):", X_train['edad'].isna().sum())
print("Ausentes en edad (test):", X_test['edad'].isna().sum())

Ausentes en edad (train): 1
Ausentes en edad (test): 0


### Ejercicio 8. Estrategias de imputación
Utilizaremos mediana para numéricas y moda para categóricas. Crea ambos objetos `SimpleImputer` **sin ajustarlos todavía**.

In [7]:
# TODO: imputer_numerico = SimpleImputer(...)
imputer_numerico = SimpleImputer(strategy='median')

# TODO: imputer_categorico = SimpleImputer(...)
imputer_categorico = SimpleImputer(strategy='most_frequent')

| Variable | Estrategia | Justificación |
|---|---|---|
| edad | Mediana | Numérica y la mediana no se deja llevar por valores raros. |
| antiguedad_meses | Mediana | Numérica, valor típico sin afectar por extremos. |
| ingresos_anuales | Mediana | Hay muchos 9.000 y valores altos, la media se movería. |
| gasto_mensual | Mediana | Asimétrica con outliers, la media sale inflada. |
| dispositivo | Moda | Categórica, no se puede hacer media. |
| zona | Moda | Categórica, se rellena con la más frecuente. |

**¿Por qué no ajustar el imputador con entrenamiento y prueba a la vez?**

Respuesta: Porque la mediana y la moda estarían calculadas con datos de prueba. El modelo "vería" algo de la prueba antes de evaluarlo (data leakage) y el resultado saldría mejor de lo que es.

### Ejercicio 9. One-Hot Encoding
Prueba `OneHotEncoder` con un DataFrame independiente. Configura `handle_unknown="ignore"` y `sparse_output=False`. Muestra matriz y nombres de columnas.

In [8]:
ejemplo = pd.DataFrame({
    "dispositivo": ["Móvil", "Ordenador", "Tablet", "Móvil"]
})

# TODO: encoder_nominal = OneHotEncoder(...)
encoder_nominal = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# TODO: aplicar fit_transform a ejemplo
matriz = encoder_nominal.fit_transform(ejemplo)
print(matriz)

# TODO: consultar get_feature_names_out()
print(encoder_nominal.get_feature_names_out())

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
['dispositivo_Móvil' 'dispositivo_Ordenador' 'dispositivo_Tablet']


**Responde:** ¿Cuántas columnas se generan? ¿Qué representan? ¿Por qué no codificar Móvil=0, Ordenador=1 y Tablet=2 como categorías ordenadas?

Respuesta: 3 columnas, una por dispositivo, con 1 si el cliente usa ese y 0 si no. No se hace Móvil=0, Ordenador=1, Tablet=2 porque no hay orden entre ellos, y el modelo pensaría que Tablet es "mayor" que Móvil o que está más lejos.

### Ejercicio 10. Codificación ordinal
Configura `OrdinalEncoder` para el orden Bronce, Plata, Oro, Platino. Comprueba su funcionamiento con un ejemplo independiente.

In [9]:
ejemplo_fidelidad = pd.DataFrame({
    "nivel_fidelidad": ["Oro", "Bronce", "Platino", "Plata"]
})

# TODO: encoder_ordinal = OrdinalEncoder(categories=[[...]])
encoder_ordinal = OrdinalEncoder(categories=[['Bronce', 'Plata', 'Oro', 'Platino']])

# TODO: transformar ejemplo_fidelidad y mostrar el resultado
ejemplo_fidelidad['codigo'] = encoder_ordinal.fit_transform(ejemplo_fidelidad).ravel()
ejemplo_fidelidad

,nivel_fidelidad,codigo
0,Oro,2.0
1,Bronce,0.0
2,Platino,3.0
3,Plata,1.0


**Responde:** ¿Qué representan los códigos? ¿La distancia entre niveles consecutivos es necesariamente igual?

Respuesta: La posición de cada nivel: Bronce 0, Plata 1, Oro 2 y Platino 3. No, el modelo ve siempre una distancia de 1, pero en la realidad pasar de Oro a Platino no tiene por qué ser lo mismo que de Bronce a Plata.

## Parte 4 - Escalado
### Ejercicio 11. Min-Max frente a Z-score
Aplica ambos escaladores al siguiente ejemplo. Redondea a tres decimales y completa la tabla.

In [10]:
datos = pd.DataFrame({"valor": [10, 20, 30, 40, 50]})

# TODO: crear MinMaxScaler y StandardScaler
minmax = MinMaxScaler()
estandar = StandardScaler()

# TODO: transformar datos con cada uno
datos_minmax = minmax.fit_transform(datos)
datos_estandar = estandar.fit_transform(datos)

# TODO: mostrar tabla comparativa
pd.DataFrame({
    'Original': datos['valor'],
    'Min-Max': datos_minmax.ravel().round(3),
    'StandardScaler': datos_estandar.ravel().round(3),
})

,Original,Min-Max,StandardScaler
0,10,0.00,-1.414
1,20,0.25,-0.707
2,30,0.50,0.000
3,40,0.75,0.707
4,50,1.00,1.414


| Original | Min-Max | StandardScaler |
|---:|---:|---:|
| 10 | 0.000 | -1.414 |
| 20 | 0.250 | -0.707 |
| 30 | 0.500 | 0.000 |
| 40 | 0.750 | 0.707 |
| 50 | 1.000 | 1.414 |

**Responde:** ¿Qué rango da Min-Max en este ejemplo? ¿Qué valor toma la media tras estandarizar? ¿Pueden aparecer valores negativos? ¿Qué escalado elegirías para un algoritmo basado en distancias y por qué?

Respuesta: Min-Max deja todo entre 0 y 1. Tras estandarizar la media es 0. Sí, con StandardScaler todo lo que está por debajo de la media sale negativo. Para distancias (KNN, por ejemplo) usaría StandardScaler, porque con outliers como los del gasto Min-Max aplasta al resto de valores en un rango muy pequeño.

### Ejercicio 12. Variables que escalaremos
Para esta práctica trataremos `satisfaccion` como numérica, aunque conceptualmente es una escala ordinal de 1 a 5. Es una simplificación que deberá revisarse en otros modelos.

In [11]:
numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales",
    "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]

# TODO: comprobar que las columnas existen en X_train
for col in numericas + nominales + ordinales:
    print(f"{col}: {col in X_train.columns}")

X_train[numericas].describe().loc[['min', 'max']].round(2)

edad: True
antiguedad_meses: True
ingresos_anuales: True
gasto_mensual: True
num_compras: True
devoluciones: True
satisfaccion: True
dispositivo: True
zona: True
nivel_fidelidad: True


,edad,antiguedad_meses,ingresos_anuales,gasto_mensual,num_compras,devoluciones,satisfaccion
min,18.0,2.0,9000.00,5.77,2.0,0.0,1.0
max,75.0,120.0,82925.37,260.00,17.0,5.0,5.0


**Responde:** ¿Qué variables tienen escalas más distintas? ¿Es obligatorio escalar para cualquier algoritmo? ¿Qué particularidad tiene `satisfaccion`?

Respuesta: `ingresos_anuales` (miles de euros) frente a `devoluciones` o `satisfaccion` (de 0 a 5). No es obligatorio para todos: los árboles de decisión no lo necesitan, pero KNN o la regresión logística sí. `satisfaccion` es en realidad una escala ordinal de 1 a 5 y aquí la tratamos como número por simplificar.

## Parte 5 - Pipeline completo
### Ejercicio 13. Pipeline numérico
Construye un `Pipeline` que impute con mediana y después estandarice.

In [12]:
# TODO: pipeline_numerico = Pipeline(steps=[
#     ("imputer", ...),
#     ("scaler", ...)
# ])
pipeline_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='median')),
    ("scaler", StandardScaler())
])

**¿Por qué imputamos antes de escalar en este pipeline?**

Respuesta: Para que el escalado se haga con todos los huecos ya rellenos y al final no quede ningún NaN. Si escaláramos primero, los NaN seguirían ahí y el modelo daría error.

### Ejercicio 14. Pipeline nominal
Imputa con moda y después aplica One-Hot Encoding (`handle_unknown="ignore"`, `sparse_output=False`).

In [13]:
# TODO: pipeline_nominal = Pipeline(steps=[
#     ("imputer", ...),
#     ("encoder", ...)
# ])
pipeline_nominal = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("encoder", OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

### Ejercicio 15. Pipeline ordinal
Imputa con moda y codifica Bronce → Plata → Oro → Platino. Gestiona categorías desconocidas con `handle_unknown="use_encoded_value"` y `unknown_value=-1`.

In [14]:
# TODO: pipeline_ordinal = Pipeline(steps=[
#     ("imputer", ...),
#     ("encoder", OrdinalEncoder(
#         categories=[[...]],
#         handle_unknown="use_encoded_value",
#         unknown_value=-1
#     ))
# ])
pipeline_ordinal = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("encoder", OrdinalEncoder(
        categories=[['Bronce', 'Plata', 'Oro', 'Platino']],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

### Ejercicio 16. ColumnTransformer
Combina los tres pipelines para aplicar cada transformación únicamente a sus columnas.

In [15]:
# TODO: preprocesador = ColumnTransformer(transformers=[
#     ("num", pipeline_numerico, numericas),
#     ("nom", pipeline_nominal, nominales),
#     ("ord", pipeline_ordinal, ordinales)
# ])
preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_numerico, numericas),
    ("nom", pipeline_nominal, nominales),
    ("ord", pipeline_ordinal, ordinales)
])

### Ejercicio 17. Transformar entrenamiento y prueba
Ajusta el preprocesador **solo** con entrenamiento y aplícalo a prueba sin reajustarlo.

In [16]:
# TODO: X_train_preparado = preprocesador.fit_transform(...)
X_train_preparado = preprocesador.fit_transform(X_train)

# TODO: X_test_preparado = preprocesador.transform(...)
X_test_preparado = preprocesador.transform(X_test)

**Responde:** ¿Por qué no usar `fit_transform()` en prueba? ¿Qué aprende `fit()`? ¿Qué ocurre si aparece una categoría nominal nueva en prueba?

Respuesta: Porque se volverían a calcular medianas, medias y categorías con la prueba, eso es data leakage y además train y test quedarían transformados con reglas distintas. `fit()` aprende las medianas y modas para imputar, la media y desviación para escalar y las categorías de cada columna. Si sale una categoría nueva, con `handle_unknown="ignore"` esa fila queda con todo ceros en las columnas de esa variable y no da error.

## Parte 6 - Validación e informe
### Ejercicio 18. Verificar matrices
Comprueba tamaños, ausentes, tipos numéricos, correspondencia con las etiquetas y nombres de características.

In [17]:
# TODO: mostrar X_train_preparado.shape y X_test_preparado.shape
print("Train:", X_train_preparado.shape)
print("Test:", X_test_preparado.shape)

# TODO: contar np.isnan(...) en ambas matrices
print("NaN en train:", np.isnan(X_train_preparado).sum())
print("NaN en test:", np.isnan(X_test_preparado).sum())

# TODO: consultar dtype y get_feature_names_out()
print("dtype:", X_train_preparado.dtype)
for i, col in enumerate(preprocesador.get_feature_names_out(), start=1):
    print(f"{i:2}. {col}")

# TODO: verificar filas de matrices frente a len(y_train), len(y_test)
print("Filas train = y_train:", X_train_preparado.shape[0] == len(y_train))
print("Filas test = y_test:", X_test_preparado.shape[0] == len(y_test))

# TODO: verificar que entrenamiento y prueba tienen igual número de columnas
print("Mismas columnas:", X_train_preparado.shape[1] == X_test_preparado.shape[1])

Train: (320, 16)
Test: (80, 16)
NaN en train: 0
NaN en test: 0
dtype: float64
 1. num__edad
 2. num__antiguedad_meses
 3. num__ingresos_anuales
 4. num__gasto_mensual
 5. num__num_compras
 6. num__devoluciones
 7. num__satisfaccion
 8. nom__dispositivo_Móvil
 9. nom__dispositivo_Ordenador
10. nom__dispositivo_Tablet
11. nom__zona_Centro
12. nom__zona_Este
13. nom__zona_Norte
14. nom__zona_Oeste
15. nom__zona_Sur
16. ord__nivel_fidelidad
Filas train = y_train: True
Filas test = y_test: True
Mismas columnas: True


**Responde:** ¿Por qué aumentaron las columnas? ¿Coinciden las características en ambos conjuntos? ¿Quedan ausentes? ¿Están alineadas las etiquetas?

Respuesta: Pasamos de 10 a 16 columnas porque One-Hot crea una por categoría (3 de dispositivo y 5 de zona). Sí, las dos tienen las mismas 16. No queda ningún ausente. Sí, cada matriz tiene las mismas filas que su `y`.

### Ejercicio 19. Reconstruir DataFrames
Convierte las matrices preparadas en `df_train_preparado` y `df_test_preparado`. Usa `preprocesador.get_feature_names_out()` y conserva los índices de X_train/X_test.

In [18]:
# TODO: columnas_finales = preprocesador.get_feature_names_out()
columnas_finales = preprocesador.get_feature_names_out()

# TODO: df_train_preparado = pd.DataFrame(..., columns=..., index=X_train.index)
df_train_preparado = pd.DataFrame(X_train_preparado, columns=columnas_finales, index=X_train.index)

# TODO: df_test_preparado = pd.DataFrame(..., columns=..., index=X_test.index)
df_test_preparado = pd.DataFrame(X_test_preparado, columns=columnas_finales, index=X_test.index)

# TODO: mostrar head() de ambos
display(df_train_preparado.head())
display(df_test_preparado.head())

,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
360,-0.068723,-1.095565,0.651045,-1.086877,0.277945,-0.185205,-0.254293,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0
356,-0.677904,1.129874,1.422884,-1.092939,0.277945,-1.069767,-0.254293,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
138,0.845048,-1.540653,0.907575,0.537957,-0.457116,-1.069767,1.495680,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
42,-1.408922,0.447406,0.476223,0.303390,1.380537,-1.069767,1.495680,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
216,-0.495150,1.456272,-0.493983,1.466735,0.645476,-1.069767,1.495680,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
388,0.053113,-1.540653,-0.994595,-0.887364,1.013006,1.583919,-0.254293,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
302,-0.495150,1.040856,-0.185324,-0.504414,-0.089586,-1.069767,-0.254293,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
21,1.697902,-1.036220,0.611217,-0.253507,-0.457116,1.583919,-0.254293,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0
368,0.784130,-1.095565,0.373354,-0.176812,-1.559708,1.583919,0.620694,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0
378,-0.312396,0.269371,-1.092628,1.067445,1.013006,-1.069767,1.495680,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
